In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image
import numpy as np
from scipy.optimize import curve_fit
from glob import glob
from scipy.signal import savgol_filter
import seaborn as sns
from pathlib import Path 
import re
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

In this notebook, you will work on the plate reader growth data of different  𝑆.𝑐𝑒𝑟𝑒𝑣𝑖𝑠𝑖𝑎𝑒 strains containing various combinations of the GAL1 and DOA1 promoter with either the GAL1 or DOA1 gene + mScarlet fluorophore exposed to different carbon source transitions.

Firstly, we want to focus on how the strain alterations affect the growth in each strain, especially compared to the wildtype strain. Some of the strains used are detailed in **panel A** of **Figure 1** below. 

The strains which are not depicted are: 

- YET784 = P<sub>Gal1</sub> - DOA1
- YET963 = P<sub>Doa1</sub> - mScarlet
  
All strains and their descriptions can be found in "Yeast strains.pdf"

- For these 2 missing strains that are not depicted, do you have a clear idea of their strains characteristics?

Secondly, we want to focus how the different carbon source transitions affect growth in every strain. The growth protocol is detailed in the word document found in this folder titled "Plate Reader Exp Set Up.pdf" and is summarized in the schematic shown below in **panel B**. The different transitions are:

'GLUGLU', 'GALGAL', 'RAFRAF', 'GLUGAL', 'GLURAF', 'RAFGAL', 'GALGLU', 'RAFGLU', 'GALRAF'. 

- Do you already have certain expectations of which transitions are faster or slower growers?

![title](imgs/SC_practical_growth_reader_schematic.png)

***

# Load and format data.

In [ ]:
#Parse the data files
files = glob(fr'Plate Reader Data\*.xlsx')
files

In [ ]:
# Load different files as a pandas dataframe using the index
df = pd.read_excel(files[1], sheet_name="Table All Cycles")

# Display the first few rows of the DataFrame
df.head(15) 

- What do you notice about the first 15 rows of the dataframe? It seems that there is some metadata that should not be included. Are you able to remove it? The resulting dataframe should only contain the raw data columns A1-F8 and the time column.

In [ ]:
# your code here: 
#initialise df_cleaned
# 1. Set the column headers based on the 12th row of the dataframe (index 11)
df_cleaned = 
df_cleaned.columns = 

# 2. Remove the first 12 rows
df_cleaned = 

# 3. Reset the index after dropping rows (otherwise the indexing of the dataframe will be off)

# 4. Drop the first column named Well*
df_cleaned = 

# 5. Rename the new first column to 'Time'


The time columns is in a strange format too. Can you convert the data to hours and minutes in the column onto two new columns named "time_hours" and "time_minutes"?

In [ ]:
#Your code here:  
df_cleaned["time_minutes"] = 
df_cleaned["time_hours"]= 

Next, we will start naming the different columns according to the conditions they belong to. Below you can find the plate layout.

![title](imgs/Plate_layout.png)

- Can you fill in the dictionary beneath to map the columns to each condition, ignoring the blanks?

In [ ]:
# Map of conditions
#These conditions are valid for the data coming from SPECTROSTAR YZMA; the following list matches the automatic columns 
#name given by the machine to each nutrient transition tested
conditions = {
    'GLUGLU': ['E1', 'E2', 'E7', 'E8'],
# fill in the rest
} 

In [ ]:
all_samples = sum(conditions.values(), [])

rename_dict = {}
for condition, samples in conditions.items():
    rename_dict.update({
        samples[0]: f"{condition}1",
        samples[1]: f"{condition}1",
        samples[2]: f"{condition}2",
        samples[3]: f"{condition}2"
    })

# Rename the columns in the DataFrame
df_renamed = df_cleaned.rename(columns=rename_dict)

# Display the DataFrame with the new column names
df_renamed.head(3)

In [ ]:
# Create a figure and set the number of subplots (48 subplots, arranged in 6 rows and 8 columns)
num_rows = 6
num_cols = 8
fig, axes = plt.subplots(num_rows, num_cols, figsize=(num_cols * 5, num_rows * 5))  #6x8 grid of subplots

# Flatten the 2D axes array to 1D for easy indexing
axes = axes.flatten()

# Collect the variable names of the renamed columns, skipping 'time','time_delta','time_in_hours' and 'time_in_minutes'
# or any column contianing something different than raw OD600 values
renamed_columns = [col for col in df_renamed.columns if col not in ['time','time_delta','time_minutes','time_hours']]

# iterate over the renamed columns and plot them below

#### YOUR CODE HERE for iterating over the columns and plotting them in the 48 different subplots ###########################









#### END CODE BLOCK #########################################################################################################

# Set the overall title of the figure
fig.suptitle('RAW DATA PLOT', fontsize=16) 

# Adjust layout to prevent overlap
plt.tight_layout() 
plt.subplots_adjust(top=0.95)   # Adjust the space for the suptitle 
plt.show()  # Display the plots 

- Look at the graph above and compare it to the plate layout. Are the global results meeting your expectations?

***

# Calculating the growth rate and the lag phase and plotting.

![title](imgs/Growth_curve_characteristics.png)

Now that you have seen the growth data we want to extract some important features from it, being the lag phase and the maximal growth rate. The maximal growth rate is determined by fitting a linear curve to the exponential part of the logged transformed growth data. After, we can determine the lag phase by finding where this fitted linear curve and the population size at timepoint zero intersect (see panel 3).

However, before we can fit a linear curve to the exponential part of the growth data, we first have to:

log transform our growth data, and remove noise in our data by using a smoothing function (panel 1).
Find the section of the graph where a linear curve should be fitted to i.e. the linear part of log transformed growth data (panel 2). In the code below we will perform these operations.

In [ ]:
#mean calculation and log transformation 
conditions_samples = {
    'GLUGLU': ['GLUGLU1', 'GLUGLU1', 'GLUGLU2', 'GLUGLU2'],
    'GALGAL': ['GALGAL1', 'GALGAL1', 'GALGAL2', 'GALGAL2'],
    'RAFRAF': ['RAFRAF1', 'RAFRAF1', 'RAFRAF2', 'RAFRAF2'],
    'GLUGAL': ['GLUGAL1', 'GLUGAL1', 'GLUGAL2', 'GLUGAL2'],
    'GLURAF': ['GLURAF1', 'GLURAF1', 'GLURAF2', 'GLURAF2'],
    'RAFGAL': ['RAFGAL1', 'RAFGAL1', 'RAFGAL2', 'RAFGAL2'],
    'GALGLU': ['GALGLU1', 'GALGLU1', 'GALGLU2', 'GALGLU2'],
    'RAFGLU': ['RAFGLU1', 'RAFGLU1', 'RAFGLU2', 'RAFGLU2'],
    'GALRAF': ['GALRAF1', 'GALRAF1', 'GALRAF2', 'GALRAF2']
}

# Initialize the df_transformed DataFrame to store log-transformed means
df_transformed = pd.DataFrame()

# Calculate the means for each condition within a sample
for condition, samples in conditions_samples.items():
    # Extract corresponding sample columns for the condition from df_final
    sample1, sample2, sample3, sample4 = samples
    sample_data = df_renamed[['time_hours', sample1, sample2, sample3, sample4]]
    
    # Ensure all columns are numeric and handle missing values (NaN)
    sample_data = sample_data.apply(pd.to_numeric, errors='coerce')
    
    # Calculate the mean of Sample1 and Sample2 for each condition, and Sample3 and Sample4 (2 technical replicates per biological replicate)
    sample_data['mean_sample1_sample2'] = sample_data[[sample1, sample2]].mean(axis=1)
    sample_data['mean_sample3_sample4'] = sample_data[[sample3, sample4]].mean(axis=1)
    
    # Log-transform the means (natural log)
    sample_data['log_mean_sample1_sample2'] = np.log(sample_data['mean_sample1_sample2'])
    sample_data['log_mean_sample3_sample4'] = np.log(sample_data['mean_sample3_sample4'])
    
    # Store the transformed means in df_transformed
    df_transformed[condition + '_log_sample1'] = sample_data['log_mean_sample1_sample2']
    df_transformed[condition + '_log_sample2'] = sample_data['log_mean_sample3_sample4'] 
    
df_transformed.head()  

Next you will plot the log transformed data.

In [ ]:
# Step 1: Add 'time_in_hours' from df_renamed to df_transformed 
df_transformed['time_hours'] = 

# Step 2: Plot all conditions for Sample1 and Sample2
# Sample 1 conditions: Collect all columns ending in '_log_sample1'
sample1_columns = 
# Sample 2 conditions: Collect all columns ending in '_log_sample2'
sample2_columns = 

# Plot setup. Initialise a plot containing two subplots: one for Sample1, one for Sample2:
fig, (ax1, ax2) = 

# Step 3: Plot all Sample 1 conditions on the first subplot
for condition in sample1_columns:


# Set y-axis and x-axis limits for Sample 1 plot

# Step 4:  Plot all Sample 2 conditions on the second subplot
for condition in sample2_columns: 

# Set y-axis and x-axis limits for Sample 2 plot


# Set the overall title


- How are the plots changed after log transformation?

- Why do we log transform the data?

In [ ]:
# save the plot
#fig.savefig("test.png", dpi=300)

Next we will apply a smoothing function to our logged transformed growth data.

for more information on the function parameters see: # added 

In [ ]:
# SAVITZKY-GOLAY FILTER APPLICATION 
from scipy.signal import savgol_filter
#savgol_filter?

In [ ]:
df_filtered = df_transformed.copy()  #create a copy to store filtered data

for col in df_transformed.columns:
    if col != "time_hours": 
        df_filtered[col] = savgol_filter(df_transformed[col], window_length=24, polyorder=1)

# Your code here plot the smoothed curves ######################################################### #



        

###################################################################################################

- If the data was not smoothed, how could this affect the determination of the maximal growth rate?

Now we will extract the maximal growth rates and the lagphase from each growth curve. We will need several functions to do this, which are all detailed below.

FUNCTIONS NEEDED FOR MAXIMAL GROWTH RATE AND LAG PHASE DETECTION 

In [ ]:
# linear function to fit
def linear_func(x, mu, c):
    return mu * x + c
    
#to extract mu, tlag and save processed values into dataframes
def extract_mu_and_tlag(df, col_name):
    # Reset index for safety
    df = df.reset_index(drop=True) 

    # Compute the derivative of the data (growth rate)
    growth_rate = np.gradient(df[col_name])
    
    # Find index of maximum growth rate/
    max_growth_rate_index = np.nanargmax(growth_rate)
    growth_rate_max = np.nanmax(growth_rate)

    # Compute window size (ws) based on doubling time formula
    ws = round(np.log(2) / (growth_rate_max))
    if ws<3:
        ws = 3
        print(f'ws was calculated to be {ws}')

    # Ensure the indices are within bounds
    start_index = max(0, max_growth_rate_index - ws)
    end_index = min(len(df) - 1, max_growth_rate_index + ws)

    # Select window around max growth rate
    xdata = df.index[start_index:end_index + 1]  # Use index for cycles
    ydata = df[col_name].iloc[start_index:end_index + 1]  # Use column values for OD600
    
    # Fit a linear curve to the selected region
    try:
        popt, _ = curve_fit(linear_func, xdata, ydata)
        mu, c = popt  # Extract slope (mu) and intercept (c)
    except RuntimeError:
        mu, c = np.nan, np.nan  # Handle fitting failure

    # Calculate the mean population at Tzero over the first 3 datapoints
    t0 = df[col_name].iloc[:3].mean()  
    # Extraxt lagphase
    tlag = (t0 - c) / mu if mu != 0 else np.nan

    # List values fitted linear curve
    y=list(linear_func(df.index, mu,c))

    # Prepare output
    output_aggregate = {
        'max_growth_rate_index': [max_growth_rate_index],
        'start_index': [start_index],
        'end_index': [end_index],
        'ws': [ws] ,
        'mu': [mu * 6] ,
        'tlag': [tlag * 0.1666], #tlag in hours,
        'tzero': [t0],
        'c': [c],
    }
    output_raw_data = {
        'smoothed_values': df[col_name].values,
        'growth_rate': growth_rate * 6,
        'fitted_linear_curve': y, 
        'cycles': df.index.values,  # Store the index as 'cycles', every cycle is 10 mins
        'time_hours': df.time_hours
    }

    return pd.DataFrame(output_aggregate), pd.DataFrame(output_raw_data )

- Where in the code is written over which area the linear curve is fitted?

In [ ]:
# APPLY THE FUNCTIONS TO THE DF_FILTERED DATAFRAME
results = []
results_raw = []

# List of columns to process (exclude 'time_in_hours')
columns_to_process = [col for col in df_filtered.columns if col != 'time_hours']

# Apply the function for each column
for col in columns_to_process:
    col_results, col_results_raw = extract_mu_and_tlag(df_filtered, col)
    col_results['column'] = col  # Store the column name for identification
    col_results_raw['column'] = col
    results.append(col_results)
    results_raw.append(col_results_raw)

# Combine all results into a single DataFrame
final_results = pd.concat(results).reset_index(drop=True)
final_results_raw = pd.concat(results_raw).reset_index(drop=True)

# Show the first few rows of the final results to inspect
final_results

# You can also have a look at the final_results raw table
# final_results_raw # what data is contained here

# Uncomment to save extracted data
#final_results.to_csv('put_name_here.csv')

We can also visualize the extraction of lagphases and growth rates by plotting the values in the final results raw dataframe.

In [ ]:
#add column as hue for seaborn
final_results_raw['transition'] = final_results_raw['column'].str.split('_',expand=True)[0]

fix,ax= plt.subplots(ncols=2, figsize=(15,5))

sns.lineplot(final_results_raw,y='growth_rate',x='time_hours', hue='transition', units='column',estimator=None,linewidth=.8,ax=ax[0])

sns.lineplot(final_results_raw,y='fitted_linear_curve',x='time_hours',color='black', units='column',estimator=None,linewidth=2,ax=ax[1],alpha=0.2)
sns.lineplot(final_results_raw,y='smoothed_values',x='time_hours', hue='transition', units='column',estimator=None,linewidth=.8,ax=ax[1])
ax[1].set_ylim(-3,.8)

- Go back to the function "extract_mu_and_tlag". Do you understand how the linear curve and the Tzero are used to determine the lagphase? 

***

- In the last part we extracted the growth rates and lagphase of the different strains. Now, we want to plot the differences in lag phases and growh rates. This part you will code yourself! Closely, inspect the plots you have created. What conclusions can be drawn? 

In [ ]:
# your code here

- Did you note that you so far only extracted the growth rates and lagphases of a single strain? Complete the same extraction for the other strains as well and make sure to compare the extracted data for all strains and transitions as well. Again it is your turn to code.

In [ ]:
# your plotting code here
# Repeat the notebook for all other files and save the CSVs. then use the hue and style variables in seaborn to plot all data.
# Make multiple plots and see which ones work best to summarise the data

Practical done! Make sure to save the figures you created for your report.